In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve()))

import torch
import config
from models.embedding import *
from models.student_model import *
from models.teacher_model import *
from training.baseline_train import *
from training.distill_train import *
from distill_loss import *
from data.create_dataloader import *
train_load, val_load, test_load = load_and_create_dataloaders()

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve()))

import torch
import config
from models.embedding import *
from models.student_model import *
from models.teacher_model import *
from training.baseline_train import *
from training.distill_train import *
from distill_loss import *
from data.create_dataloader import *
if __name__ == '__main__':
    train_load, val_load, test_load = load_and_create_dataloaders()
    
    hist = train_distillation(
        teacher=teacher_model(),
        student=create_student('custom'),
        train_loader=train_load,
        val_loader=val_load,
        test_loader = test_load,
        teacher_name='cifar10_resnet56',
        student_name='custom',
        epochs=5,
        alpha = 0.6
    )

In [ ]:

CHECKPOINT_DIR = Path("saved_models")
CHECKPOINT_DIR.mkdir(exist_ok=True)

train_loader, val_loader, test_loader = load_and_create_dataloaders()

teacher = teacher_model()

# 3. Student factories
students = {
    "resnet20": lambda: create_student("resnet20"),
    "resnet32": lambda: create_student("resnet32"),
    "cnn_small": lambda: create_student('custom')
}

shared_kwargs = {
    "epochs": config.NUM_EPOCHS,
    "lr": config.LEARNING_RATE,
    "train_loader": train_loader,
    "val_loader": val_loader,
    "test_loader": test_loader
}

trained_models = {}

for name, factory in students.items():
    print(f"\n{'='*40}")
    print(f"STUDENT: {name.upper()}")
    
    print("Distill")
    distill_model = factory().to(config.DEVICE)
    train_distillation(
        teacher=teacher, student=distill_model, student_name=name,
        alpha=0.6, distill_type="cosine", **shared_kwargs
    )
    # Baseline
    print("Baseline")
    baseline_model = factory().to(config.DEVICE)
    train_baseline(model=baseline_model, model_name=name, **shared_kwargs)
    trained_models[f"{name}_baseline"] = baseline_model
    torch.save(baseline_model.state_dict(), CHECKPOINT_DIR / f"{name}_baseline.pth")
    
    # Distillation
    trained_models[f"{name}_distill"] = distill_model
    torch.save(distill_model.state_dict(), CHECKPOINT_DIR / f"{name}_distill.pth")

print("done")

Using cache found in /home/peter007/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master



STUDENT: RESNET20
Distill


Using cache found in /home/peter007/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 0 | Train Loss: 0.7792 | Acc: 55.86% | Cls_loss:  1.46 | distill_loss: 0.33
Epoch 1 | Train Loss: 0.5587 | Acc: 65.10% | Cls_loss:  1.05 | distill_loss: 0.23
Epoch 2 | Train Loss: 0.4640 | Acc: 71.78% | Cls_loss:  0.88 | distill_loss: 0.19
Epoch 3 | Train Loss: 0.4082 | Acc: 75.18% | Cls_loss:  0.77 | distill_loss: 0.17
Epoch 4 | Train Loss: 0.3687 | Acc: 75.74% | Cls_loss:  0.70 | distill_loss: 0.15
Epoch 5 | Train Loss: 0.3432 | Acc: 76.28% | Cls_loss:  0.65 | distill_loss: 0.14
Epoch 6 | Train Loss: 0.3201 | Acc: 78.78% | Cls_loss:  0.61 | distill_loss: 0.13
Epoch 7 | Train Loss: 0.3051 | Acc: 80.28% | Cls_loss:  0.58 | distill_loss: 0.12
Epoch 8 | Train Loss: 0.2906 | Acc: 81.06% | Cls_loss:  0.55 | distill_loss: 0.12
Epoch 9 | Train Loss: 0.2748 | Acc: 82.36% | Cls_loss:  0.52 | distill_loss: 0.11
Epoch 10 | Train Loss: 0.2639 | Acc: 82.14% | Cls_loss:  0.50 | distill_loss: 0.11
Epoch 11 | Train Loss: 0.2578 | Acc: 81.80% | Cls_loss:  0.49 | distill_loss: 0.11
Epoch 12 | Tra

In [2]:
print(validate(teacher_model(), val_load))

Using cache found in /home/peter007/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


(0.0007411844181442114, 100.0)


In [1]:
hist = train_distillation(teacher_model(), 
create_student('resnet20'), 
train_load, 
val_load, 
epochs = 30, 
teacher_name = 'cifar10_resnet56', 
student_name = 'resnet20'
)
print(hist)

NameError: name 'train_distillation' is not defined

/media/winD/Wstęp do ML/proj1/venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:433: UserWarning: Got pickle error when attempting to start a worker Process. This might be because the worker Process arguments are not picklable. Python 3.14+ changed the multiprocessing start method in non-Mac POSIX platforms to 'forkserver', which requires the worker Process arguments to be picklable. You can also try multiprocessing.set_start_method('fork').
  return _MultiProcessingDataLoaderIter(self)


PicklingError: Can't pickle local object <function TinyImageNet.__init__.<locals>.<lambda> at 0x7fe021c52c40>